# Fix the dicom files

In [ ]:
from tqdm import tqdm
from pathlib import Path

import numpy as np
import pandas as pd

import nibabel as nib
from nibabel.processing import resample_from_to
import pydicom

import json

In [43]:
root_dir = Path('/data/vision/polina/projects/fetal/common-data/BRAIN-IQA/dataset1/')
save_loc = Path('/data/vision/polina/users/marcusbl/data')

label_map = {
            '{"image_quality":"bad"}': 0,
            '{"image_quality":"good"}': 1
} 

In [ ]:
for person_path in list(root_dir.iterdir()):

    if not person_path.is_dir():
        continue

    for stack_path in person_path.iterdir():
        if not stack_path.is_dir(): # not a stack folder
            continue

        # 1) Parse CSV File for Labels
        csv_file = next(stack_path.glob("*.csv"), None)
        if csv_file is None:
            continue

        label_df = pd.read_csv(csv_file)

        # 2) Align Nifti & Dicom Orders
        dicoms_dir_path = stack_path / "dicoms"
        nifti_scan_path = stack_path / 'converted.nii.gz'
        nifti_mask_path = stack_path / 'converted_mask.nii.gz'

        nifti_scan = nib.load(nifti_scan_path)
        nifti_mask = nib.load(nifti_mask_path)
        
        # 3) Fix Ordering of Dicom Files
        sorted_dicom_files = sorted(dicoms_dir_path.glob("*.dcm"), key = lambda s: s.name[:4], reverse=True)
        first_dicom = pydicom.dcmread(sorted_dicom_files[0]).pixel_array.astype(dtype=np.float32)
        first_nifti =  np.rot90(np.asarray(nifti_scan.dataobj[:, :, 0]), k = 1, axes = (0,1))

        if not np.allclose(first_dicom, first_nifti): # don't match -> reverse order
            sorted_dicom_files.reverse()

        # 4) Create Aligned Dicom & Nifti w/ Masks  
        dicom_stack = np.stack([pydicom.dcmread(dcm_file).pixel_array.astype(dtype=np.float32) for dcm_file in sorted_dicom_files], axis = -1)
        nifti_stack = nifti_scan.get_fdata(dtype=np.float32)
        mask_stack = resample_from_to(nifti_mask, nifti_scan, order = 0).get_fdata(dtype=np.float32)
        
        # rot each by 90 degrees
        nifti_stack = np.rot90(nifti_stack, k=1, axes=(0,1))
        mask_stack = np.rot90(mask_stack, k=1, axes=(0,1))

        assert np.allclose(dicom_stack, nifti_stack)
        
        # 5) Create Labeled JSON
        dicom_labels = {}
        old_mapping = {}
        for scan_num, fpath in enumerate(sorted_dicom_files):
            # Get the label
            row = label_df.loc[label_df["External ID"] == (fpath.stem + ".png")]
            label_str = row["Label"].values[0]

            # If unknown label -> skip
            if label_str not in label_map: 
                continue
            
            dicom_labels[scan_num] = label_map[label_str]
            old_mapping[scan_num] = str(fpath)

        if len(dicom_labels) == 0:
            continue

        # 6) Save Outputs
        clean_folder = save_loc / '/'.join(stack_path.parts[-2:]) / 'clean'
        clean_folder.mkdir(exist_ok=True, parents=True)

        with open(clean_folder / 'labels.json', 'w') as f:
            json.dump(dicom_labels, f)
        with open(clean_folder / 'old_maps.json', 'w') as f:
            json.dump(old_mapping, f)

        np.save(clean_folder / 'dicoms.npy', dicom_stack)
        np.save(clean_folder / 'niftis.npy', nifti_stack)
        np.save(clean_folder / 'masks.npy', mask_stack)
        
        print(f"Saved to {clean_folder}")
        exit


Loading People Data:   0%|          | 0/44 [00:00<?, ?it/s]

Saved to /data/vision/polina/users/marcusbl/data/anon-00014-scan-3/stack_23/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00014-scan-3/stack_20/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00014-scan-3/stack_18/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00014-scan-3/stack_15/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00014-scan-3/stack_17/clean


Loading People Data:   2%|▏         | 1/44 [00:02<01:28,  2.06s/it]

Saved to /data/vision/polina/users/marcusbl/data/anon-00014-scan-3/stack_19/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00068-7-17-2013/stack_6/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00068-7-17-2013/stack_1/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00068-7-17-2013/stack_5/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00068-7-17-2013/stack_8/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00068-7-17-2013/stack_2/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00068-7-17-2013/stack_7/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00068-7-17-2013/stack_3/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00068-7-17-2013/stack_9/clean


Loading People Data:   5%|▍         | 2/44 [00:05<01:55,  2.75s/it]

...

Saved to /data/vision/polina/users/marcusbl/data/anon-00052-8-17-2017/stack_2/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00048-9-25-2013/stack_6/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00048-9-25-2013/stack_1/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00048-9-25-2013/stack_5/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00048-9-25-2013/stack_2/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00048-9-25-2013/stack_3/clean


Loading People Data:  14%|█▎        | 6/44 [00:13<01:22,  2.16s/it]

Saved to /data/vision/polina/users/marcusbl/data/anon-00048-9-25-2013/stack_4/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_5/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_15/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_18/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_2/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_12/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_20/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_16/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_6/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_11/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_1/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_13/cl

Loading People Data:  16%|█▌        | 7/44 [00:37<05:38,  9.16s/it]

Saved to /data/vision/polina/users/marcusbl/data/anon-00090-3-1-2018/stack_17/clean
Saved to /data/vision/polina/users/marcusbl/data/anon-00103-4-4-2014/stack_19/clean
